### Importing Necessary Liberaries

In [141]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                            AdaBoostClassifier,
                            GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (accuracy_score,
                            precision_score,
                            recall_score,
                            classification_report,
                            confusion_matrix,
                            f1_score,
                            roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE

### Importing the dataset

In [49]:
raw = pd.read_csv('Customer-Churn-Records.csv')
df = raw.copy()
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


### Dropping Unnecessary Columns

In [52]:
cols_to_drop = ['RowNumber','CustomerId','Surname','Complain']
df.drop(columns=cols_to_drop,inplace=True)

### Spliting into Train and Test

In [55]:
X = df.drop('Exited',axis = 1)
y = df['Exited']

In [57]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.25,random_state=12
)

### Importing Preprocessor File

In [60]:
with open('preprocessor.pkl','rb') as file:
    preprocessor = pickle.load(file)

In [62]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

### Model Tranining 

In [163]:
def performance_evaluation(true,predicted):
    acc_score = accuracy_score(true,predicted)
    precision = precision_score(true,predicted)
    recall = recall_score(true,predicted)
    f1 = f1_score(true,predicted)
    matrix = confusion_matrix(true,predicted)
    report = classification_report(true,predicted)
    return acc_score,precision,recall,f1,matrix,report

In [81]:
## Beginning Model Training
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'CatBoost': CatBoostClassifier(),
    'Random Forest' :RandomForestClassifier(),
    'AdaBoost':AdaBoostClassifier(),
    'Gradient Boosting': GradientBoostingClassifier()
}
results= []
for name, model in models.items():
    model.fit(X_train,y_train)

    ## Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    ## model Evaluations
    train_acc_score, train_precision_score, train_recall_score, train_f1_score ,train_confusion_matrix, train_report = performance_evaluation(y_train,y_train_pred)
    test_acc_score, test_precision_score, test_recall_score, test_f1_score, test_confusion_matrix, test_report = performance_evaluation(y_test,y_test_pred)
    
    print(name)
    
    print('Model performance for Training set')
    print("- Accuracy Score: {:.4f}".format(train_acc_score))
    print('- Precision Score: {:.4f}'.format(train_precision_score))
    print('- Recall Score: {:.4f}'.format(train_recall_score))
    print('- F1 Score: {:.4f}'.format(train_f1_score))
    print('- Confusion Matrix: ',(train_confusion_matrix))
    print('- Classification Report : \n',(train_report))

    print('-'*50)

    print('Model performance for Testing set')
    print("- Accuracy Score: {:.4f}".format(test_acc_score))
    print('- Precision Score: {:.4f}'.format(test_precision_score))
    print('- Recall Score: {:.4f}'.format(test_recall_score))
    print('- F1 Score: {:.4f}'.format(test_f1_score))
    print('- Confusion Matrix: ',(test_confusion_matrix))
    print('- Classification Report : \n',(test_report))

    print('='*50)
    print('\n')

    results.append({
    'Model': name,
    'Test Accuracy': test_acc_score,
    'Test Precision': test_precision_score,
    'Test Recall': test_recall_score,
    'Test F1 Score': test_f1_score
    })

Logistic Regression
Model performance for Training set
- Accuracy Score: 0.8147
- Precision Score: 0.5950
- Recall Score: 0.1943
- F1 Score: 0.2930
- Confusion Matrix:  [[5822  196]
 [1194  288]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.83      0.97      0.89      6018
           1       0.60      0.19      0.29      1482

    accuracy                           0.81      7500
   macro avg       0.71      0.58      0.59      7500
weighted avg       0.78      0.81      0.77      7500

--------------------------------------------------
Model performance for Testing set
- Accuracy Score: 0.8016
- Precision Score: 0.6613
- Recall Score: 0.2212
- F1 Score: 0.3315
- Confusion Matrix:  [[1881   63]
 [ 433  123]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.81      0.97      0.88      1944
           1       0.66      0.22      0.33       556

    accuracy                   

In [85]:
results

[{'Model': 'Logistic Regression',
  'Test Accuracy': 0.8016,
  'Test Precision': 0.6612903225806451,
  'Test Recall': 0.22122302158273383,
  'Test F1 Score': 0.33153638814016173},
 {'Model': 'Decision Tree',
  'Test Accuracy': 0.7908,
  'Test Precision': 0.5309568480300187,
  'Test Recall': 0.5089928057553957,
  'Test F1 Score': 0.519742883379247},
 {'Model': 'CatBoost',
  'Test Accuracy': 0.8604,
  'Test Precision': 0.8071216617210683,
  'Test Recall': 0.4892086330935252,
  'Test F1 Score': 0.6091825307950728},
 {'Model': 'Random Forest',
  'Test Accuracy': 0.8584,
  'Test Precision': 0.8435374149659864,
  'Test Recall': 0.4460431654676259,
  'Test F1 Score': 0.5835294117647059},
 {'Model': 'AdaBoost',
  'Test Accuracy': 0.8408,
  'Test Precision': 0.7379518072289156,
  'Test Recall': 0.44064748201438847,
  'Test F1 Score': 0.5518018018018018},
 {'Model': 'Gradient Boosting',
  'Test Accuracy': 0.8596,
  'Test Precision': 0.8213166144200627,
  'Test Recall': 0.4712230215827338,
  'Tes

### Model Comparison

- Logistic Regression underperformed due to its low Recall and F1 Score.
- Decision Tree achieved the highest Recall but at the cost of lower Precision and Accuracy.
- CatBoost achieved the highest F1 Score (0.609), providing the best balance between Precision and Recall.
- Gradient Boosting and Random Forest also performed well but were slightly behind CatBoost in terms of F1 Score.
- Therefore, CatBoost and RandomForest were selected for further hyperparameter tuning.

### Hyperparameter Tunning

In [110]:
rf_params = {
    "max_depth": [5, 8, 15, 10],
    "max_features": [5,6, 7, 8],
    "min_samples_split": [2, 8, 15, 20],
    "n_estimators": [100, 200, 500, 1000]
}
cb_params = {
    "iterations": [100, 200, 300],
    "learning_rate": [0.03, 0.05,0.75],
    "depth": [2,3,4,5],
    "l2_leaf_reg": [1, 3, 5, 7],
    "subsample": [0.8, 1.0]
}
gb_params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 4, 5],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "subsample": [0.8, 1.0]
}

In [112]:
randomcv_models = [
    ('Random Forest', RandomForestClassifier(),rf_params),
    ('CatBoost' , CatBoostClassifier(),cb_params),
    ('Gradient Boosting' , GradientBoostingClassifier(),gb_params)
]

In [114]:
model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1,
                                   random_state=12)
    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
0:	learn: 0.6545421	total: 7.17ms	remaining: 1.43s
1:	learn: 0.6253646	total: 15ms	remaining: 1.48s
2:	learn: 0.5985352	total: 21.2ms	remaining: 1.39s
3:	learn: 0.5716911	total: 27.8ms	remaining: 1.36s
4:	learn: 0.5505579	total: 35.3ms	remaining: 1.38s
5:	learn: 0.5331239	total: 41.8ms	remaining: 1.35s
6:	learn: 0.5142509	total: 48.6ms	remaining: 1.34s
7:	learn: 0.4997366	total: 54.3ms	remaining: 1.3s
8:	learn: 0.4865860	total: 61.4ms	remaining: 1.3s
9:	learn: 0.4751506	total: 67.5ms	remaining: 1.28s
10:	learn: 0.4628160	total: 74.7ms	remaining: 1.28s
11:	learn: 0.4544649	total: 82.8ms	remaining: 1.3s
12:	learn: 0.4445728	total: 89.9ms	remaining: 1.29s
13:	learn: 0.4372792	total: 96.1ms	remaining: 1.28s
14:	learn: 0.4293822	total: 102ms	remaining: 1.26s
15:	learn: 0.4247846	total: 108ms	remaining: 1.24s
16:	learn: 0.4199192	total: 113ms	remaining: 1.22s
17:	lear

In [147]:
## Beginning Model Training
models = {
    'Random Forest' :RandomForestClassifier(
        n_estimators= 500,
        min_samples_split= 2,
        max_features= 7,
        max_depth= 15,
        random_state=12),
     'CatBoost': CatBoostClassifier(subsample= 0.8,
                                    learning_rate= 0.05,
                                    l2_leaf_reg= 5,
                                    iterations= 200,
                                    depth= 3,
                                    random_state=12),
    'Gradient Boosting' : GradientBoostingClassifier(
        subsample = 0.8,
        n_estimators = 200,
        min_samples_split = 2,
        min_samples_leaf = 4,
        max_depth = 3,
        learning_rate = 0.05,
        random_state=12
    )
}
new_results= []
for name, model in models.items():
    model.fit(X_train,y_train)

    ## Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    ## model Evaluations
    train_acc_score, train_precision_score, train_recall_score, train_f1_score, train_confusion_matrix, train_report = performance_evaluation(y_train,y_train_pred)
    test_acc_score, test_precision_score, test_recall_score, test_f1_score, test_confusion_matrix, test_report = performance_evaluation(y_test,y_test_pred)
    
    print(name)
    
    print('Model performance for Training set')
    print("- Accuracy Score: {:.4f}".format(train_acc_score))
    print('- Precision Score: {:.4f}'.format(train_precision_score))
    print('- Recall Score: {:.4f}'.format(train_recall_score))
    print('- F1 Score: {:.4f}'.format(train_f1_score))
    print('- Confusion Matrix: ',(train_confusion_matrix))
    print('- Classification Report : \n',(train_report))

    print('-'*50)

    print('Model performance for Testing set')
    print("- Accuracy Score: {:.4f}".format(test_acc_score))
    print('- Precision Score: {:.4f}'.format(test_precision_score))
    print('- Recall Score: {:.4f}'.format(test_recall_score))
    print('- F1 Score: {:.4f}'.format(test_f1_score))
    print('- Confusion Matrix: ',(test_confusion_matrix))
    print('- Classification Report : \n',(test_report))

    print('='*50)
    print('\n')

    new_results.append({
    'Model': name,
    'Test Accuracy': test_acc_score,
    'Test Precision': test_precision_score,
    'Test Recall': test_recall_score,
    'Test F1 Score': test_f1_score
    })

Random Forest
Model performance for Training set
- Accuracy Score: 0.9828
- Precision Score: 1.0000
- Recall Score: 0.9130
- F1 Score: 0.9545
- ROC AUC Score: 0.9565
- Confusion Matrix:  [[6018    0]
 [ 129 1353]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.98      1.00      0.99      6018
           1       1.00      0.91      0.95      1482

    accuracy                           0.98      7500
   macro avg       0.99      0.96      0.97      7500
weighted avg       0.98      0.98      0.98      7500

--------------------------------------------------
Model performance for Testing set
- Accuracy Score: 0.8600
- Precision Score: 0.8323
- Recall Score: 0.4640
- F1 Score: 0.5958
- ROC AUC Score: 0.7186
- Confusion Matrix:  [[1892   52]
 [ 298  258]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.86      0.97      0.92      1944
           1       0.83      0.46      0.60 

In [108]:
new_results

[{'Model': 'Random Forest',
  'Test Accuracy': 0.8608,
  'Test Precision': 0.8443708609271523,
  'Test Recall': 0.45863309352517984,
  'Test F1 Score': 0.5944055944055944},
 {'Model': 'CatBoost',
  'Test Accuracy': 0.86,
  'Test Precision': 0.8259493670886076,
  'Test Recall': 0.4694244604316547,
  'Test F1 Score': 0.5986238532110092}]

### Using SMOTE for balancing data

In [127]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [131]:
X_train_smote.shape,y_train_smote.shape

((12036, 25), (12036,))

In [133]:
X_train.shape

(7500, 25)

In [174]:
## Beginning Model Training
models = {
    'Random Forest' :RandomForestClassifier(
        n_estimators= 500,
        min_samples_split= 2,
        max_features= 7,
        max_depth= 15,
        random_state=12),
     'CatBoost': CatBoostClassifier(subsample= 0.8,
                                    learning_rate= 0.05,
                                    l2_leaf_reg= 5,
                                    iterations= 200,
                                    depth= 3,
                                    random_state=12),
    'Gradient Boosting' : GradientBoostingClassifier(
        subsample = 0.8,
        n_estimators = 200,
        min_samples_split = 2,
        min_samples_leaf = 4,
        max_depth = 3,
        learning_rate = 0.05,
        random_state=12
    )
}
new_results= []
for name, model in models.items():
    model.fit(X_train_smote,y_train_smote)

    ## Make Predictions
    y_train_smote_pred = model.predict(X_train_smote)
    y_test_smote_pred = model.predict(X_test)
    
    ## model Evaluations
    train_acc_score, train_precision_score, train_recall_score, train_f1_score, train_confusion_matrix, train_report = performance_evaluation(y_train_smote,y_train_smote_pred)
    test_acc_score, test_precision_score, test_recall_score, test_f1_score, test_confusion_matrix, test_report = performance_evaluation(y_test,y_test_smote_pred)
    
    print(name)
    
    print('Model performance for Training set')
    print("- Accuracy Score: {:.4f}".format(train_acc_score))
    print('- Precision Score: {:.4f}'.format(train_precision_score))
    print('- Recall Score: {:.4f}'.format(train_recall_score))
    print('- F1 Score: {:.4f}'.format(train_f1_score))
    print('- Confusion Matrix: ',(train_confusion_matrix))
    print('- Classification Report : \n',(train_report))

    print('-'*50)

    print('Model performance for Testing set')
    print("- Accuracy Score: {:.4f}".format(test_acc_score))
    print('- Precision Score: {:.4f}'.format(test_precision_score))
    print('- Recall Score: {:.4f}'.format(test_recall_score))
    print('- F1 Score: {:.4f}'.format(test_f1_score))
    print('- Confusion Matrix: ',(test_confusion_matrix))
    print('- Classification Report : \n',(test_report))

    print('='*50)
    print('\n')

    new_results.append({
    'Model': name,
    'Test Accuracy': test_acc_score,
    'Test Precision': test_precision_score,
    'Test Recall': test_recall_score,
    'Test F1 Score': test_f1_score
    })

Random Forest
Model performance for Training set
- Accuracy Score: 0.9899
- Precision Score: 0.9905
- Recall Score: 0.9894
- F1 Score: 0.9899
- Confusion Matrix:  [[5961   57]
 [  64 5954]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      6018
           1       0.99      0.99      0.99      6018

    accuracy                           0.99     12036
   macro avg       0.99      0.99      0.99     12036
weighted avg       0.99      0.99      0.99     12036

--------------------------------------------------
Model performance for Testing set
- Accuracy Score: 0.8572
- Precision Score: 0.7287
- Recall Score: 0.5701
- F1 Score: 0.6398
- Confusion Matrix:  [[1826  118]
 [ 239  317]]
- Classification Report : 
               precision    recall  f1-score   support

           0       0.88      0.94      0.91      1944
           1       0.73      0.57      0.64       556

    accuracy                         

In [175]:
new_results

[{'Model': 'Random Forest',
  'Test Accuracy': 0.8572,
  'Test Precision': 0.728735632183908,
  'Test Recall': 0.5701438848920863,
  'Test F1 Score': 0.6397578203834511},
 {'Model': 'CatBoost',
  'Test Accuracy': 0.854,
  'Test Precision': 0.732360097323601,
  'Test Recall': 0.5413669064748201,
  'Test F1 Score': 0.6225439503619442},
 {'Model': 'Gradient Boosting',
  'Test Accuracy': 0.8516,
  'Test Precision': 0.70509977827051,
  'Test Recall': 0.5719424460431655,
  'Test F1 Score': 0.631578947368421}]

### Choosing Final Model 

In [178]:
final_model = RandomForestClassifier(
    n_estimators= 500,
    min_samples_split= 2,
    max_features= 7,
    max_depth= 15,
    random_state=12
)

final_model.fit(X_train_smote, y_train_smote)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",15
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",7
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",12
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: boo

In [180]:
with open('model.pkl','wb') as file:
    pickle.dump(final_model,file)

## Final Model Evaluation

- Random Forest was selected as the final model after hyperparameter tuning and applying SMOTE.
- The model achieved an F1 Score of **0.6398** on the test set, outperforming the other evaluated models.
- The model correctly identified **57%** of customers who churned while maintaining a precision of **72.9%**.
- The difference between training and testing performance indicates some overfitting, which is expected after applying SMOTE and using a high-capacity ensemble model.
- Overall, the model provides a good balance between identifying churning customers and limiting false alarms, making it suitable for customer retention strategies.